[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C60_Edge_Deployment_Consistency_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（三层框架 / 二分定位 / 容差表 / 排查顺序）

目标：把本课的**方法论**变成可运行的代码，而不是几张 PPT 上的框图。

本 notebook 你会亲手实现：
1. **一致性度量与容差表**——把「fp16 噪声 / 定点舍入 / 语义错误」三档量级实际算出来，
   证明它们之间隔着一个数量级，因此阈值是可推导的而不是拍脑袋的
2. **三层鸿沟的「指纹」分类器**——给定一个 diff 张量，自动判断错误属于哪一类
3. **逐层二分定位器**——在一条 12 阶段的管线上用 3 次探针找到首个分歧算子，
   并验证探针次数 = ⌈log₂(n+1)⌉
4. **排查顺序的最优性**——证明按「概率 ÷ 成本」降序排查使期望总成本最小
5. **误差放大模型**——输入端 1e-3 的差异如何在 8 层网络里变成分数扰动，
   再变成排序翻转，最后变成 mAP 掉点
6. **环境自检**

> 心智模型：**不要问「为什么模型效果差」，要问「第一个对不上的张量在哪」。**
> 前者没有终点，后者最多 ⌈log₂ n⌉ 步。

## 1 · 一致性度量与容差表

先把三档差异的量级算出来。归一化用 mmdet 的经典配置
`mean=[123.675, 116.28, 103.53]`、`std=[58.395, 57.12, 57.375]`，
「tensor 单位」指归一化之后的张量（模型真正吃到的东西）。

In [ ]:
import numpy as np, math, sys, platform, time, json
rng = np.random.default_rng(60)
np.set_printoptions(precision=4, suppress=True)

MEAN = np.array([123.675, 116.28, 103.53])
STD  = np.array([58.395, 57.12, 57.375])

# 一张「自然图」：低频结构 + 细纹理（细纹理是后面混叠实验的关键）
yy, xx = np.mgrid[0:240, 0:240]
img = np.clip(np.stack([110 + 40*np.sin(xx/47.), 118 + 35*np.cos(yy/39.),
                        125 + 30*np.sin((xx+yy)/61.)], -1)
              + 18*np.sin(2*np.pi*xx/5.)[..., None], 0, 255)
ten = (img - MEAN) / STD                      # 模型真正吃到的张量

def report(a, b, name):
    d = np.abs(a - b)
    return dict(name=name, max=float(d.max()), mean=float(d.mean()),
                p99=float(np.percentile(d, 99)))

# ① fp16 的表示误差（不可避免）
fp16 = report(ten.astype(np.float32), ten.astype(np.float16).astype(np.float32), 'fp16 表示误差')
# ② 定点 resize 的舍入：uint8 输出四舍五入，误差在 ±0.5 灰阶内均匀分布
err_fix = rng.uniform(-0.5, 0.5, ten.shape) / STD.mean()
fixed = report(ten, ten + err_fix, '定点舍入 ±0.5 灰阶')
# ③ 语义错误：INTER_LINEAR vs INTER_AREA 的典型差（模块 01 会实测出 21.4 灰阶）
sem = report(ten, ten + 21.4/STD.mean(), 'resize 语义错误')

print(f"{'差异来源':<22s}{'max':>12s}{'mean':>12s}")
for r in (fp16, fixed, sem):
    print(f"{r['name']:<22s}{r['max']:>12.6f}{r['mean']:>12.6f}")

ratio_impl = fixed['max'] / fp16['max']
ratio_sem  = sem['mean']  / fixed['max']
print(f"\n定点舍入(max) / fp16(max)     = {ratio_impl:6.1f} 倍")
print(f"语义错误(mean) / 定点舍入(max) = {ratio_sem:6.1f} 倍")
assert fp16['max'] < 2e-3,  fp16
assert 8e-3 < fixed['max'] < 9e-3 and fixed['mean'] < 5e-3, fixed
assert sem['mean'] > 0.3, sem
assert ratio_sem > 20, '语义错误必须比实现噪声大一个数量级以上，阈值才有空档可放'
print('\n✅ 三档之间隔着一到两个数量级 —— 所以容差是可推导的，不是拍脑袋的。')

In [ ]:
# 把「阈值放在空档里」这件事写成一个判定器
TOL = dict(preproc_max=0.02, preproc_mean=0.005)   # tensor 单位

def verdict(a, b, tol=TOL):
    """预处理阶段的一致性判定：返回 (PASS/FAIL, 细节)"""
    d = np.abs(np.asarray(a) - np.asarray(b))
    ok = (d.max() <= tol['preproc_max']) and (d.mean() <= tol['preproc_mean'])
    return ('PASS' if ok else 'FAIL'), dict(max=float(d.max()), mean=float(d.mean()))

cases = [
    ('fp16 抖动（不该报警）',      ten.astype(np.float16).astype(np.float32)),
    ('定点舍入 ±0.5 灰阶（不该报）', ten + err_fix),
    ('align_corners 差异（该报）', ten + 0.038),
    ('resize 语义错（该报）',      ten + 21.4/STD.mean()),
]
print(f"{'场景':<26s}{'判定':>6s}{'max':>10s}{'mean':>10s}")
got = []
for name, other in cases:
    v, det = verdict(ten, other)
    got.append(v)
    print(f'{name:<26s}{v:>6s}{det["max"]:>10.4f}{det["mean"]:>10.4f}')
assert got == ['PASS', 'PASS', 'FAIL', 'FAIL'], got
print('\n✅ 同一组阈值同时做到：放过实现噪声、拦住语义错误。')
print('⚠️  最窄的一道缝：定点舍入的 max(=0.0086) 与 align_corners(=0.038) 只差 4.4 倍，')
print('   阈值 max<=0.02 正好卡在中间 —— 这就是「阈值要有来源」的含义。')

## 2 · diff 的形状就是指纹

找到分歧阶段后不要急着读代码，**先看 diff 张量长什么样**。
每一类预处理错误在张量空间里都有特征性的支撑集与频谱：

| diff 的样子 | 原因 |
|---|---|
| 只在边缘一圈非零 | `align_corners` / padding |
| 高频条纹、棋盘格 | **resize 插值方式不同（混叠）** |
| 通道置换后归零 | **BGR/RGB 弄反** |
| 全图常数偏移 | mean 不同 |
| 全图按比例缩放 | std 不同 / 少除了 255 |
| 零散无结构、幅值 1e-3 | 正常数值噪声 |

下面把这个「用眼睛认」的过程写成代码。

In [ ]:
def fingerprint(a, b, edge=2, tol=2e-3):
    """给定两侧张量，返回最可能的错误类型。a=参考(训练侧)，b=待查(部署侧)。"""
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = b - a
    if np.abs(d).max() <= tol:
        return 'noise'                                   # 在噪声档内，不是 bug
    # 通道置换？把 b 的通道反序后再比
    if a.ndim == 3 and a.shape[-1] == 3 and np.abs(b[..., ::-1] - a).max() <= tol:
        return 'channel_swap'
    # 边缘带 vs 内部：能量是否集中在边界
    mask = np.zeros(a.shape[:2], bool)
    mask[:edge] = mask[-edge:] = True; mask[:, :edge] = mask[:, -edge:] = True
    de = np.abs(d[mask]).mean(); di = np.abs(d[~mask]).mean()
    if di < 1e-9 or de / max(di, 1e-12) > 20:
        return 'border'
    # 常数偏移？（逐通道去均值后残差极小）
    ax = tuple(range(d.ndim - 1)) if a.ndim == 3 else None
    if np.abs(d - d.mean(axis=ax, keepdims=True)).max() < 0.05 * np.abs(d).max() + 1e-9:
        return 'const_shift'
    # 比例缩放？（b/a 近似常数）
    r = b / np.where(np.abs(a) < 1e-6, np.nan, a)
    if np.nanstd(r) < 0.02 * abs(np.nanmean(r)):
        return 'scale'
    # 高频结构？（相邻差分的能量占比高）
    hf = np.abs(np.diff(d, axis=1)).mean() / (np.abs(d).mean() + 1e-12)
    return 'high_freq' if hf > 0.8 else 'unknown'

base = ten.copy()
tests = {
    'noise':        base + rng.normal(0, 3e-4, base.shape),
    'channel_swap': base[..., ::-1],
    'const_shift':  base + 0.31,
    'scale':        base * 1.17,
    'high_freq':    base + 0.4*np.sign(np.sin(2*np.pi*xx/2.))[..., None],
}
bd = base.copy(); bd[:2] += 0.4; bd[-2:] += 0.4; bd[:, :2] += 0.4; bd[:, -2:] += 0.4
tests['border'] = bd

for k, v in tests.items():
    got = fingerprint(base, v)
    print(f'{k:<14s} -> {got}')
    assert got == k, (k, got)
print('\n✅ 6 种指纹全部识别正确 —— 这就是「看图认」的代码化。')

## 3 · 逐层二分定位

关键性质：`D(k) = 1[前 k 阶段已分歧]` 关于 k **单调**——
分歧一旦产生就不会自己愈合。单调布尔序列上找第一个 1 = 二分查找，
代价 ⌈log₂(n+1)⌉ 次探针。

In [ ]:
def bisect_first_divergence(probe, n):
    """probe(k) -> bool，表示「跑到第 k 阶段为止是否已分歧」（k 从 1 到 n）。
       返回 (首个分歧阶段或 None, 探针次数)。
       技巧：把「全程无分歧」编码成虚拟位置 n+1（D(n+1) 定义为 True），
       于是不需要先单独探一次 probe(n) —— 省下的这一次让上界正好等于 ⌈log2(n+1)⌉。"""
    lo, hi, calls = 0, n + 1, 0      # 不变式：D(lo)=False（lo=0 天然成立），D(hi)=True
    while hi - lo > 1:
        mid = (lo + hi) // 2
        calls += 1
        if probe(mid):
            hi = mid
        else:
            lo = mid
    return (None if hi == n + 1 else hi), calls

# 12 阶段管线，分歧首次出现在第 7 阶段（resize）
STAGES = ['read', 'decode', 'to_rgb', 'crop_roi', 'cast_f32', 'clip',
          'resize', 'letterbox', 'normalize', 'hwc2chw', 'batch', 'contiguous']
FIRST_BAD = 7
def probe_factory(first_bad, counter):
    def probe(k):
        counter[0] += 1
        return k >= first_bad
    return probe

cnt = [0]
k, calls = bisect_first_divergence(probe_factory(FIRST_BAD, cnt), len(STAGES))
print(f'首个分歧阶段 = 第 {k} 个 = {STAGES[k-1]!r}   探针次数 = {calls}')
assert k == FIRST_BAD and calls == cnt[0]
assert calls <= math.ceil(math.log2(len(STAGES) + 1)), (calls,)
print(f'理论上界 ⌈log2({len(STAGES)}+1)⌉ = {math.ceil(math.log2(len(STAGES)+1))}')

# 全部位置都验一遍：正确性 + 探针次数上界
worst = 0
for fb in range(1, len(STAGES) + 1):
    kk, cc = bisect_first_divergence(probe_factory(fb, [0]), len(STAGES))
    assert kk == fb, (fb, kk)
    worst = max(worst, cc)
    assert cc <= math.ceil(math.log2(len(STAGES) + 1)), (fb, cc)
kk, cc = bisect_first_divergence(lambda k: False, len(STAGES))
assert kk is None and cc == math.ceil(math.log2(len(STAGES) + 1)), (kk, cc)
print(f'12 个位置全部定位正确；最坏探针次数 = {worst}（顺序扫描最坏要 {len(STAGES)} 次）')
print('✅ 12 阶段 → 4 次；1000 阶段 → 10 次。这就是「30 分钟」的复杂度依据。')

In [ ]:
# 真实一点：两条管线各自跑，probe 用「张量 diff 是否超容差」实现
def pipeline(x, resize_kernel='area'):
    """极简 6 阶段管线；resize 用 1D 盒滤波 / 抽样两种实现来制造分歧。"""
    out = {}
    out['cast'] = x.astype(np.float64)
    out['clip'] = np.clip(out['cast'], 0, 255)
    s = 2
    if resize_kernel == 'area':                       # 2x2 盒平均
        out['resize'] = out['clip'].reshape(x.shape[0]//s, s, x.shape[1]//s, s, 3).mean((1, 3))
    else:                                             # 抽样（= 无抗锯齿）
        out['resize'] = out['clip'][::s, ::s]
    out['normalize'] = (out['resize'] - MEAN) / STD
    out['chw'] = np.moveaxis(out['normalize'], -1, 0)
    out['batch'] = out['chw'][None]
    return out

ORDER = ['cast', 'clip', 'resize', 'normalize', 'chw', 'batch']
tr = pipeline(img, 'area')          # 训练侧
dp = pipeline(img, 'sample')        # 部署侧（resize 实现不同）

def probe_tensors(k, tol=0.02):
    name = ORDER[k-1]
    a, b = np.asarray(tr[name], float), np.asarray(dp[name], float)
    if a.shape != b.shape:
        return True                  # 形状都不同，必然分歧
    return bool(np.abs(a - b).max() > tol * (STD.mean() if name in ('cast','clip','resize') else 1))

k, calls = bisect_first_divergence(probe_tensors, len(ORDER))
print(f'首个分歧阶段 = 第 {k} 个 = {ORDER[k-1]!r}，探针 {calls} 次')
assert ORDER[k-1] == 'resize'
fp = fingerprint(tr['normalize'], dp['normalize'])
print(f'该阶段 diff 的指纹 = {fp!r}')
print(f'diff: max={np.abs(tr["normalize"]-dp["normalize"]).max():.4f}  '
      f'mean={np.abs(tr["normalize"]-dp["normalize"]).mean():.4f}')
assert fp in ('high_freq', 'unknown')
print('\n✅ 二分 + 指纹 = 「第 3 个算子 resize，高频结构 → 插值方式不同」。')
print('   从看到现象到给出结论，全程不需要读一行两侧的源码。')

## 4 · 排查顺序：为什么「概率 ÷ 成本」降序是最优的

第 1 节给了根因的概率分布。但排查顺序不该只看概率——**还要看成本**。
把每个候选原因 i 的「命中概率 pᵢ」和「检查成本 cᵢ」放在一起，
期望总成本在按 **pᵢ/cᵢ 降序** 排查时最小（这是经典的调度不等式 / Smith rule）。

In [ ]:
CAUSES = [
    # (名字, 命中概率 p, 检查成本 c 小时)
    ('预处理不一致',      0.40, 0.5),
    ('后处理/坐标变换',   0.25, 0.5),
    ('数值精度/量化',     0.15, 4.0),
    ('评测口径不一致',    0.10, 2.0),
    ('真实域差(要闭环)',  0.10, 40.0),
]

def expected_cost(order):
    """按 order 顺序逐个检查，期望总成本 = Σ_j c_j * P(前 j-1 个都没命中)"""
    tot, remain = 0.0, 1.0
    for name, p, c in order:
        tot += remain * c            # 无论命中与否，这一项的检查成本都要付
        remain -= p                  # 命中就停
    return tot

by_ratio = sorted(CAUSES, key=lambda z: -z[1]/z[2])
by_prob  = sorted(CAUSES, key=lambda z: -z[1])
by_cost  = sorted(CAUSES, key=lambda z:  z[2])
worst    = sorted(CAUSES, key=lambda z:  z[1]/z[2])

print(f"{'策略':<22s}{'期望总成本(小时)':>18s}")
for nm, o in [('p/c 降序（最优）', by_ratio), ('只按概率降序', by_prob),
              ('只按成本升序', by_cost), ('p/c 升序（最差）', worst)]:
    print(f'{nm:<22s}{expected_cost(o):>18.3f}')

# 穷举验证 p/c 降序确实是最优
import itertools
best = min(itertools.permutations(CAUSES), key=expected_cost)
assert abs(expected_cost(best) - expected_cost(by_ratio)) < 1e-12
assert expected_cost(by_ratio) <= expected_cost(by_prob) + 1e-12
print('\n穷举 120 种顺序，最优顺序 =')
for i, (n, p, c) in enumerate(by_ratio, 1):
    print(f'  {i}. {n:<18s} p={p:.2f} c={c:>4.1f}h  p/c={p/c:>5.2f}')
print(f'\n✅ 最优 {expected_cost(by_ratio):.2f}h vs 最差 {expected_cost(worst):.2f}h，'
      f'差 {expected_cost(worst)/expected_cost(by_ratio):.1f} 倍。')
# 「只按成本升序」在这组数上恰好也最优 —— 因为这里概率与成本刚好负相关。这是巧合，不能依赖：
toy = [('A 贵但极可能', 0.90, 2.0), ('B 便宜但几乎不可能', 0.02, 0.1)]
assert expected_cost(sorted(toy, key=lambda z: -z[1]/z[2])) < expected_cost(sorted(toy, key=lambda z: z[2]))
print(f'   反例：{[t[0] for t in toy]} —— 按成本升序 {expected_cost(sorted(toy,key=lambda z:z[2])):.3f}h'
      f' > 按 p/c 降序 {expected_cost(sorted(toy,key=lambda z:-z[1]/z[2])):.3f}h。'
      '「先做便宜的」一般不是最优，只有在 p/c 也高时才是。')
print('⚠️  注意「真实域差」概率不低（10%）但成本极高（40h），永远排最后 ——')
print('   「先怀疑模型」之所以是错的，不是因为它不可能，而是因为它最贵。')

## 5 · 误差放大：1e-3 的输入差怎么变成 mAP 掉点

一致性问题让人低估的原因是：**输入端的差异看起来很小**。
但网络是有增益的——逐层的 Lipschitz 常数相乘，输入 δ 变成输出 L·δ；
分数扰动会改变**排序**，而 AP 只依赖排序。

In [ ]:
def amplify(delta_in, layer_gains):
    """逐层放大：返回每层之后的差异上界"""
    out, d = [], delta_in
    for g in layer_gains:
        d *= g
        out.append(d)
    return np.array(out)

GAINS = np.array([1.4, 1.3, 1.6, 1.2, 1.5, 1.3, 1.1, 1.2])   # 8 层的经验增益
L = float(np.prod(GAINS))
print(f"{'差异来源':<16s}{'输入 δ':>10s}{'输出 δ':>12s}{'超过 fp16 档的倍数':>20s}")
base_out = None
for name, d0 in [('fp16 噪声', 5e-4), ('align_corners', 3.8e-2), ('resize 语义错', 0.37)]:
    out = amplify(d0, GAINS)[-1]
    base_out = out if base_out is None else base_out
    print(f'{name:<16s}{d0:>10.4g}{out:>12.4g}{out/base_out:>20.1f}')
assert 8.9 < L < 9.1, L
print(f'\n整网增益 L = Π gᵢ = {L:.2f}（8 层，每层 1.1~1.6）')
print('⚠️  这是**上界**不是实际值 —— 但它说明方向：输入端的差异到输出端不会变小。')
print('   而下一段会看到：输出端 δ 落在 0.05 量级就足以改变 AP。')

In [ ]:
# 分数扰动 -> 排序翻转 -> AP 掉点
def ap_from_scores(scores, labels):
    """标准 VOC all-point AP。labels: 1=TP, 0=FP"""
    o = np.argsort(-scores, kind='stable')
    l = np.asarray(labels)[o]
    tp, fp = np.cumsum(l), np.cumsum(1 - l)
    npos = max(int(l.sum()), 1)
    rec, pre = tp / npos, tp / np.maximum(tp + fp, 1e-9)
    mr = np.concatenate([[0], rec, [1]]); mp = np.concatenate([[0], pre, [0]])
    for i in range(len(mp) - 2, -1, -1):
        mp[i] = max(mp[i], mp[i + 1])
    k = np.where(mr[1:] != mr[:-1])[0]
    return float(((mr[k+1] - mr[k]) * mp[k+1]).sum())

r2 = np.random.default_rng(7)
N = 400
labels = (r2.random(N) < 0.35).astype(int)
scores = np.clip(r2.beta(2.5, 2.0, N) + 0.25*labels, 0, 1)   # TP 分数系统性更高
ap0 = ap_from_scores(scores, labels)

print(f"{'输出端扰动 σ':>14s}{'AP':>9s}{'ΔAP':>9s}{'top-50 排序翻转率':>20s}")
prev = 1.0
for sigma in [0.0, 0.005, 0.02, 0.05, 0.12, 0.30]:
    s = scores + r2.normal(0, sigma, N)
    ap = ap_from_scores(s, labels)
    top0, top1 = set(np.argsort(-scores)[:50]), set(np.argsort(-s)[:50])
    flip = 1 - len(top0 & top1) / 50
    print(f'{sigma:>14.3f}{ap:>9.3f}{ap-ap0:>9.3f}{flip:>19.1%}')
    assert ap <= prev + 1e-9, 'AP 应随扰动单调下降（同一随机流下）'
    prev = ap
print('\n✅ 关键：AP 只依赖**排序**。分数抖动 0.05 就能换掉 top-50 里的一批，')
print('   而 0.05 在张量空间里是一个「看起来很小」的数。')
print('⚠️  这解释了为什么「张量 diff 只有 0.04，应该没影响吧」是错的直觉。')

## 6 · 环境自检

In [ ]:
t0 = time.time()
print('Python  :', sys.version.split()[0])
print('平台    :', platform.platform())
print('numpy   :', np.__version__)
ok = True
try:
    a = np.arange(12.).reshape(3, 4)
    assert np.allclose(a @ a.T, np.einsum('ij,kj->ik', a, a))
    assert np.percentile(np.arange(101.), 99) == 99.0
    _ = np.random.default_rng(0).beta(2, 2, 5)          # Generator API
except Exception as e:                                  # pragma: no cover
    ok = False; print('❌', e)
print('numpy 自检 :', 'OK' if ok else 'FAIL')
print('本课全部 notebook 纯 numpy + 标准库，CPU 即可，无需 GPU / TensorRT / 联网。')
print(f'耗时 {time.time()-t0:.3f}s')
assert ok

## ✏️ 练习 1：带上下界的二分定位器

实现 `bisect_range(probe, lo, hi)`：已知 `D(lo)=False`、`D(hi)=True`，
在开区间 `(lo, hi]` 里找首个分歧位置，返回 `(位置, 探针次数)`。
**要求**：探针次数 ≤ `ceil(log2(hi-lo))`，且不重复调用同一个 k。

In [ ]:
def bisect_range(probe, lo, hi):
    # TODO: 二分不变式 —— D(lo)=False, D(hi)=True，缩到 hi-lo==1 为止
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
for n in [1, 2, 3, 7, 12, 33, 1000]:
    for fb in ({1, n} | set(np.random.default_rng(n).integers(1, n+1, min(n, 8)).tolist())):
        seen = []
        k, c = bisect_range(lambda x: (seen.append(x), x >= fb)[1], 0, n)
        assert k == fb, (n, fb, k)
        assert len(seen) == len(set(seen)), '不能重复探同一个 k'
        assert c <= math.ceil(math.log2(n)) or n == 1, (n, fb, c)
k, c = bisect_range(lambda x: x >= 7, 4, 12)     # 已知前 4 阶段没问题
assert k == 7 and c <= 3, (k, c)
print(f'1000 阶段管线最坏探针次数 ≤ {math.ceil(math.log2(1000))}')
print('✅ 练习 1 通过：知道下界能省探针 —— 现实中「前几步肯定没问题」是常见的先验。')

## ✏️ 练习 2：最优排查顺序

实现 `optimal_order(causes)`：输入 `[(名字, p, c), ...]`，
返回按**期望总成本最小**排列的列表。（提示：Smith rule，按 p/c 降序。）
再实现 `expected_cost_of(causes)` 返回该顺序下的期望成本。

In [ ]:
def optimal_order(causes):
    # TODO
    raise NotImplementedError

def expected_cost_of(causes):
    # TODO: 用 optimal_order 排完之后算期望成本
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
o = optimal_order(CAUSES)
assert [x[0] for x in o] == [x[0] for x in by_ratio], o
assert abs(expected_cost_of(CAUSES) - expected_cost(by_ratio)) < 1e-12
# 手算小例子：A(p=.5,c=1) 与 B(p=.4,c=.2) -> B 的 p/c=2.0 > A 的 0.5，B 先
toy = [('A', 0.5, 1.0), ('B', 0.4, 0.2)]
assert [x[0] for x in optimal_order(toy)] == ['B', 'A']
# B先: 0.2*1 + 1.0*(1-0.4) = 0.8 ; A先: 1.0*1 + 0.2*(1-0.5) = 1.1
assert abs(expected_cost_of(toy) - 0.8) < 1e-12, expected_cost_of(toy)
assert abs(expected_cost(toy) - 1.1) < 1e-12
# 穷举校验
import itertools as it
assert abs(expected_cost_of(CAUSES) - min(map(expected_cost, it.permutations(CAUSES)))) < 1e-12
print('最优顺序:', [x[0] for x in o])
print(f'期望成本 {expected_cost_of(CAUSES):.3f}h')
print('✅ 练习 2 通过：「先查预处理」不是经验之谈，是 p/c 排序的结论。')

## ✏️ 练习 3：分阶段容差判定器

实现 `stage_verdict(a, b, kind)`，`kind ∈ {'preproc', 'model_fp16', 'model_int8', 'boxes'}`：

- `'preproc'`：`max ≤ 0.02` 且 `mean ≤ 0.005`
- `'model_fp16'`：相对误差 `|a-b|/(|b|+1e-3)` 的 **99 分位** `≤ 0.02`
- `'model_int8'`：同上但阈值 `0.10`
- `'boxes'`：a、b 是 `(N,4)` 框数组——**个数必须相等**，且逐框 IoU 全 `> 0.99`

返回 `'PASS'` 或 `'FAIL'`。

In [ ]:
def stage_verdict(a, b, kind):
    # TODO: 不同阶段用不同的语义空间去比 —— 这是本练习唯一的考点
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
x = ten
assert stage_verdict(x, x.astype(np.float16).astype(np.float32), 'preproc') == 'PASS'
assert stage_verdict(x, x + 0.038, 'preproc') == 'FAIL'
y = rng.normal(0, 1, (2, 64, 40, 40))
assert stage_verdict(y, y * 1.005, 'model_fp16') == 'PASS'
assert stage_verdict(y, y * 1.05,  'model_fp16') == 'FAIL'
assert stage_verdict(y, y * 1.05,  'model_int8') == 'PASS'
B1 = np.array([[10., 10., 50., 50.], [100., 100., 140., 160.]])
assert stage_verdict(B1, B1 + 0.01, 'boxes') == 'PASS'
assert stage_verdict(B1, B1 + 1.0,  'boxes') == 'FAIL'     # 1px 偏移 -> IoU 约 0.95
assert stage_verdict(B1, B1[:1],    'boxes') == 'FAIL'     # **个数不等，直接 FAIL**
print('✅ 练习 3 通过：比什么，比怎么比更重要。')
print('   框的个数不等几乎总是 NMS 语义不同，而不是数值问题 —— 不要用容差去「容忍」它。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bisect_range(probe, lo, hi):
    calls = 0
    while hi - lo > 1:
        mid = (lo + hi) // 2
        calls += 1
        if probe(mid):
            hi = mid
        else:
            lo = mid
    return hi, calls

In [ ]:
# 练习 2 参考答案
def optimal_order(causes):
    return sorted(causes, key=lambda z: -z[1] / z[2])

def expected_cost_of(causes):
    tot, remain = 0.0, 1.0
    for _, p, c in optimal_order(causes):
        tot += remain * c
        remain -= p
    return tot

In [ ]:
# 练习 3 参考答案
def _iou_rows(a, b):
    x1 = np.maximum(a[:, 0], b[:, 0]); y1 = np.maximum(a[:, 1], b[:, 1])
    x2 = np.minimum(a[:, 2], b[:, 2]); y2 = np.minimum(a[:, 3], b[:, 3])
    it = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    aa = (a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]); bb = (b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1])
    return it / np.maximum(aa + bb - it, 1e-9)

def stage_verdict(a, b, kind):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if kind == 'boxes':
        if a.shape != b.shape:
            return 'FAIL'
        return 'PASS' if (len(a) == 0 or _iou_rows(a, b).min() > 0.99) else 'FAIL'
    if a.shape != b.shape:
        return 'FAIL'
    if kind == 'preproc':
        d = np.abs(a - b)
        return 'PASS' if (d.max() <= 0.02 and d.mean() <= 0.005) else 'FAIL'
    thr = {'model_fp16': 0.02, 'model_int8': 0.10}[kind]
    rel = np.abs(a - b) / (np.abs(b) + 1e-3)
    return 'PASS' if np.percentile(rel, 99) <= thr else 'FAIL'

---
## 🧪 真实工程胶囊：一页纸的「上车掉点」排查单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 「离线好、上车差」排查单 —— 按 p/c 顺序，不要跳步
# ══════════════════════════════════════════════════════════════════════

# ── 第 0 步（10 分钟，必做）：先确认这两个数可比 ──────────────────────
#  · 车上的「0.6」是怎么得到的？同一套 GT？同一个 IoU 阈值？同一个类别表？
#  · 离线评测有没有用到车上没有的东西（GT crop / 时序平滑 / 更大分辨率）？
#  · 如果口径不同 —— **先对齐口径，这一步经常直接结案**。

# ── 第 1 步（30 分钟）：预处理对拍  p=0.40  c=0.5h  ★ 最先查 ★ ────────
#  1) 选黄金样本：不要用「正常」图。要 (a) 含 <16px 小目标 (b) 含过曝区
#     (c) 奇数尺寸 (d) 纯色大面积 (e) 极暗帧。10~30 张固定入库。
#  2) 两侧 dump：
#       np.save(f'dump/{side}_{i:02d}_{stage}.npy', tensor)
#     阶段名两边必须一致：read/to_rgb/resize/letterbox/normalize/chw/batch
#  3) 二分 + 指纹：见本 notebook 第 3、2 节
#  4) 容差（tensor 单位）：max<=0.02, mean<=0.005
#  常见结论：INTER_LINEAR vs INTER_AREA / align_corners / BGR / letterbox 变体
#  → 展开见 **模块 01**

# ── 第 2 步（30 分钟）：后处理与坐标对拍  p=0.25  c=0.5h ───────────────
#  · sigmoid/softmax 是否做了两遍或一遍都没做（看分数分布：全在 0.5 附近 = 做了两遍）
#  · 框整体偏移固定像素 = letterbox 逆变换忘了减 pad
#  · 框个数不等 = NMS 语义不同（class-wise vs agnostic / top-k 截断 / IoU 的 +1）
#  · 判据：框个数相等 & 逐框 IoU>0.99 & 类别一致
#  → 展开见 **模块 04**

# ── 第 3 步（0.5 天）：评测口径  p=0.10  c=2h ─────────────────────────
#  · 用**完全相同的图片列表**在两侧各跑一遍，比对最终框；差异应为 0
#  · 若两侧框一致但指标不同 —— 问题在评测代码，不在模型

# ── 第 4 步（0.5~1 天）：数值精度  p=0.15  c=4h ──────────────────────
#  · fp32 engine 先跑通并与 PyTorch 对齐（p99 相对误差 < 2%），**再**开 fp16
#  · fp16 掉点 -> 查溢出（激活值 > 65504）；int8 掉点 -> 查校准集分布
#  · 逐层敏感度分析 -> 混合精度
#  → 展开见 **模块 02 / 03**

# ── 第 5 步（数周）：真实域差  p=0.10  c=40h ─────────────────────────
#  · 只有前四步全绿才走到这里。此时才允许说「模型不够好」。
#  · 触发 -> 挖掘 -> 标注 -> 训练 -> 切片评测 -> 门禁（见 C58）

# ── 固化：把第 1、2 步做成 CI ────────────────────────────────────────
#   pytest tests/test_preproc_parity.py     # 黄金样本 x 逐阶段 x 容差
#   pytest tests/test_postproc_parity.py    # 逆变换 / NMS 语义 / 框个数
#   每次改预处理、换 OpenCV 版本、改 C++ 管线，都必须跑。
#   **一致性不是靠小心，是靠门禁。**
'''
print(RECIPE)
for token in ['黄金样本', 'INTER_AREA', 'align_corners', 'letterbox 逆变换',
              '框个数', '65504', 'CI', 'p=0.40']:
    assert token in RECIPE, token
print('✅ 排查单覆盖：口径对齐 / 预处理 / 后处理 / 评测 / 精度 / 域差 / CI 固化')

### 小结

- **「离线好、上车差」里约 65% 是确定性的软件缺陷**（预处理 40% + 后处理 25%），
  几行代码就能修；真正「模型不够好」只占 10%，但它最贵（40h vs 0.5h）。
  **按 p/c 降序排查，期望成本比最差顺序低数倍**——这是可证明的，不是经验之谈。
- **一致性是可验证的工程属性**：需要①可比较的中间产物 ②有来源的容差
  ③自动化判定 ④黄金样本集。缺任何一个，它都会随时间退化。
- **二分定位的复杂度是 ⌈log₂(n+1)⌉**：12 阶段 4 次探针，1000 阶段 10 次。
  依据是 `D(k)` 单调——分歧一旦产生不会自己愈合。
- **diff 的形状就是指纹**：边缘带→padding/align_corners；高频条纹→插值方式；
  通道置换→BGR/RGB；常数偏移→mean；比例缩放→std；无结构 1e-3→正常噪声。
- **容差要有来源**：fp16≈1e-3、定点舍入≈8.6e-3、align_corners≈3.8e-2、
  resize 语义错≈0.37。**三档之间隔着一到两个数量级，阈值放在空档里（0.02）。**
- **别拿 mAP 当一致性检查**：它灵敏度不够（中等错误可能只掉 2–3 点，
  淹没在种子方差里），而且归因不了。**张量对拍是门禁，端到端指标是验收。**
- **AP 只依赖排序**：输出分数抖动 0.05 就能换掉 top-50 里的一批框——
  所以「张量只差 0.04，应该没影响」是错的直觉。

下一站：**模块 01 · 预处理一致性** —— 40% 的问题都在那里，而 resize 是头号杀手。